In [1]:
# config.py

#--------Packages
from pathlib import Path

### Below from SMM notes
import pandas as pd
import matplotlib.pyplot as plt
import dask
from dask import delayed, compute
from dask.distributed import Client
import scipy.optimize as optimize
from scipy.optimize import minimize
import scipy.stats as stats
import plotly.express as px
import numpy as np
import sys
from functools import lru_cache
import warnings
import numpy as np
import pandas as pd
from scipy.optimize import minimize


!{sys.executable} -m pip install quantecon
import quantecon as qe
from quantecon.markov.approximation import tauchen

#---------relative path configuration

# Root = folder containing config.py

from pathlib import Path

try:
    ROOT = Path(__file__).resolve().parent.parent
except NameError:
    ROOT = Path.cwd()

SCRIPTS = ROOT / "scripts"
RESULTS = ROOT / "results"

RESULTS.mkdir(exist_ok=True)

print("Setup successful")
print(f"ROOT: {ROOT}")
print(f"SCRIPTS: {SCRIPTS}")
print(f"RESULTS: {RESULTS}")



Setup successful
ROOT: /Users/marlifirillo/CompEcon_Fall25/DynamicProgramming/ProblemSet8
SCRIPTS: /Users/marlifirillo/CompEcon_Fall25/DynamicProgramming/ProblemSet8/scripts
RESULTS: /Users/marlifirillo/CompEcon_Fall25/DynamicProgramming/ProblemSet8/results


In [2]:
#Module 1: Define the model parameters and set up targets moments from Table 3.

#--------CE Calibrated Parameters

beta = 0.95      # discount factor
delta = 0.15     # depreciation rate
p = 1.0          # price of capital

#--------SMM starting parameters

alpha = 0.6956
gamma = 0.1331
rho = 0.0976
sigma = 0.8932
phi_tilde0 = 0.0


#Psi d - AKA target-moments

target_moments = [
    0.03,   # a1 - coefficient on expected average Q
    0.24,   # a2 - coefficient on the profit rate
    0.40,   # serial correlation of investment rate
    0.25,   # std of profit rate (pi/K)
    3.00,   # average Q
    0.25    # fraction externally financed
]


# Parameter names used in SMM estimation
PARAMETER_NAMES = [
    "alpha",
    "gamma",
    "rho",
    "sigma",
    "phi_tilde0"
]

# Six moments matched between simulated and actual data
MOMENT_NAMES = [
    "a1",
    "a2",
    "sc(I/K)",
    "std(CF/K)",
    "mean Q",
    "external fraction"
]

#target_moments = np.asarray(target_moments, dtype=float)

# weighting_matrix = np.eye(len(target_moments))  # Weighting matrix will be an identity matrix for simplicity



In [3]:
# Module 3: Define functions for use in later steps

# 1. Profit function 
def profit(K, A, alpha):
    """
    Calculate firm profit given capital K and productivity A.
    """
    return A * K**alpha

# 2. Investment function 
def investment(K, K_next, delta):
    """
    Calculate gross investment needed to move from K to K_next.
    """
    return K_next - (1 - delta) * K

# 3. Adjustment costs now use GROSS investment from Equation (4)
def adjustment_costs(K, K_next, delta, gamma):
    """
    Calculate adjustment costs using gross investment from Equation (4).
    """
    I = investment(K, K_next, delta)
    return (gamma / 2) * (I / K) ** 2 * K


# Calculate current period net cash flow
def current_payout(K, A, alpha, K_next, delta, gamma):
    """
    Calculate current period net cash flow after investment and adjustment costs.
    """
    return (
        profit(K, A, alpha)
        - investment(K, K_next, delta)
        - adjustment_costs(K, K_next, delta, gamma)
    )

In [4]:
#Module 4: Create grids

# =============================================================================
# GRID GENERATION FUNCTIONS
# =============================================================================
def make_A_process(rho, sigma, nA=5):
    # QuantEcon tauchen method wrapper
    mc = tauchen(n=nA, rho=rho, sigma=sigma)
    logA_grid = np.asarray(mc.state_values)
    P = np.asarray(mc.P)

    # Ensure transition probabilities are valid
    P = np.where(P < 0, 0.0, P)
    P = P / P.sum(axis=1, keepdims=True)
    A_grid = np.exp(logA_grid)

    return A_grid, P

def make_K_grid(K_min=1.0, K_max=1000.0, nK=300):
    return np.linspace(K_min, K_max, nK)



In [5]:
# Module 4a: Settings for simulation and standard error exercise
# Settings used for simulation and inference
INFERENCE_CONFIG = dict(
    K_min=1.0,
    K_max=1000.0,
    nK=100,
    nA=5,
    num_firms=500,
    num_periods=100,
    burn_in=30,
    delta=float(delta),
    beta=float(beta),
    phi1=0.0
)

# Number of simulated panels used for each SMM objective
S_ESTIMATION = 1

# Number of independent panels used to estimate moment covariance
R_COVARIANCE = 100

# Number of panels used to reduce noise in numerical derivatives
S_DERIVATIVE = 10

# Seeds make simulated results reproducible
ESTIMATION_SEEDS = tuple(
    12345 + s for s in range(S_ESTIMATION)
)

DERIVATIVE_SEEDS = tuple(
    54321 + s for s in range(S_DERIVATIVE)
)

COVARIANCE_SEEDS = tuple(
    90000 + r for r in range(R_COVARIANCE)
)

# Bounds for each parameter during estimation
INFERENCE_BOUNDS = np.array([
    (0.40, 0.95),
    (0.001, 2.0),
    (0.001, 0.98),
    (0.05, 1.5),
    (0.0, 2.0)
])

# Step sizes used to calculate numerical derivatives
FD_STEPS = np.array([
    0.005,
    0.01,
    0.01,
    0.02,
    0.02
])

# Maximum number of times the optimizer can evaluate the objective
MAX_FUNCTION_EVALUATIONS = 1000

In [ ]:
def solve_inference_dp(theta, cfg):
    """Vectorized evaluation of the uploaded Bellman equation; no changed costs."""
    a, g, r, s, f = np.asarray(theta, dtype=float)
    A, P = make_A_process(r, s, nA=cfg["nA"])
    K = make_K_grid(cfg["K_min"], cfg["K_max"], nK=cfg["nK"])
    k = K[:, None, None]
    kp = K[None, :, None]
    az = A[None, None, :]
    pi = profit(k, az, a)
    inv = investment(k, kp, cfg["delta"])
    # Use exactly the same branch condition as the uploaded code, including equality.
    external = kp > pi + (1.0 - cfg["delta"]) * k
    net = pi - inv - adjustment_costs(k, kp, cfg["delta"], g)
    payoff = net - np.where(external, f + cfg["phi1"] * (inv - pi), 0.0)
    V = np.zeros((len(K), len(A)))
    for iteration in range(1, 1001):
        rhs = payoff + cfg["beta"] * (V @ P.T)[None, :, :]
        policy = np.argmax(rhs, axis=1)
        new = np.take_along_axis(rhs, policy[:, None, :], axis=1)[:, 0, :]
        error = np.max(np.abs(new - V))
        V = new
        if error < 1e-6:
            break
    else:
        raise ValueError("Inference DP did not converge; inference cannot use an unsolved policy.")
    fin = np.take_along_axis(external, policy[:, None, :], axis=1)[:, 0, :].astype(int)
    return K, A, P, V, policy, fin


def inference_panel_moments(theta, solved, seed, cfg):
    """One independent panel; same initial states, Q timing and moment definitions."""
    a, g, r, s, f = np.asarray(theta, dtype=float)
    K, A, P, V, policy, fin = solved
    N, T, B = cfg["num_firms"], cfg["num_periods"], cfg["burn_in"]
    if N < 2 or not 0 <= B < T - 1:
        raise ValueError("Need at least two firms and two retained periods.")
    uniforms = np.random.default_rng(seed).random((N, T))
    ik = np.full(N, len(K) // 2, dtype=int)
    ia = np.full(N, len(A) // 2, dtype=int)
    rates, cash, expected_q, q, financing = [np.empty((N, T - B)) for _ in range(5)]
    cumulative = np.cumsum(P, axis=1)
    cumulative[:, -1] = 1.0
    for t in range(T):
        ikp = policy[ik, ia]
        k, kp, az = K[ik], K[ikp], A[ia]
        if t >= B:
            j = t - B
            rates[:, j] = investment(k, kp, cfg["delta"]) / k
            cash[:, j] = profit(k, az, a) / k
            expected_q[:, j] = np.sum(P[ia, :] * V[ikp, :], axis=1) / kp
            q[:, j] = V[ik, ia] / k
            financing[:, j] = fin[ik, ia]
        ik = ikp
        ia = np.sum(uniforms[:, t, None] > cumulative[ia, :], axis=1)
    X = np.column_stack((np.ones(rates.size), expected_q.ravel(), cash.ravel()))
    if np.linalg.matrix_rank(X) < 3:
        raise ValueError("Simulated investment regression is rank deficient.")
    _, a1, a2 = np.linalg.lstsq(X, rates.ravel(), rcond=None)[0]
    lag, lead = rates[:, :-1].ravel(), rates[:, 1:].ravel()
    if min(np.std(lag), np.std(lead)) < 1e-12:
        raise ValueError("Investment serial correlation is undefined at this parameter trial.")
    moments = np.array([a1, a2, np.corrcoef(lag, lead)[0, 1],
                        np.std(cash), np.mean(q), np.mean(financing)])
    if not np.isfinite(moments).all():
        raise ValueError("Non-finite simulated moments.")
    return moments


def inference_mean_moments(theta, seeds, cfg=INFERENCE_CONFIG):
    """Average moments over specified panels, solving the DP once per theta."""
    solved = solve_inference_dp(theta, cfg)
    return np.mean([inference_panel_moments(theta, solved, seed, cfg) for seed in seeds], axis=0)

In [7]:
def calculate_moments(panel):
    """
    Calculate the six moments used for SMM from the simulated panel.

    Uses the simulated investment, cash flow, Q, and financing data
    to calculate the moments matched to the actual data.
    """
    investment_rate = panel["investment_rate"]
    cash_flow = panel["cash_flow"]
    expected_Q = panel["expected_Q"]
    Q = panel["Q"]
    financing = panel["financing"]

    # Regression of investment on expected Q and cash flow
    X = np.column_stack((
        np.ones(investment_rate.size),
        expected_Q.ravel(),
        cash_flow.ravel()
    ))

    if np.linalg.matrix_rank(X) < 3:
        raise ValueError(
            "Simulated investment regression is rank deficient."
        )

    regression_results = np.linalg.lstsq(
        X,
        investment_rate.ravel(),
        rcond=None
    )[0]

    _, a1, a2 = regression_results

    # Calculate serial correlation of investment
    investment_lag = investment_rate[:, :-1].ravel()
    investment_lead = investment_rate[:, 1:].ravel()

    if min(
        np.std(investment_lag),
        np.std(investment_lead)
    ) < 1e-12:
        raise ValueError(
            "Investment serial correlation is undefined at this parameter trial."
        )

    # Calculate six simulated moments
    moments = np.array([
        a1,
        a2,
        np.corrcoef(
            investment_lag,
            investment_lead
        )[0, 1],
        np.std(cash_flow),
        np.mean(Q),
        np.mean(financing)
    ])

    if not np.isfinite(moments).all():
        raise ValueError(
            "Non-finite simulated moments."
        )

    return moments


def inference_panel_moments(theta, solved, seed, cfg):
    """
    Simulate one panel and calculate its six moments.
    """
    panel = simulate_panel(
        theta,
        solved,
        seed,
        cfg
    )

    return calculate_moments(panel)


def inference_mean_moments(theta, seeds, cfg=INFERENCE_CONFIG):
    """
    Calculate average simulated moments across panels using different seeds.

    The model is solved once for each parameter set and the resulting
    policy function is used for each simulated panel.
    """
    solved = solve_inference_dp(theta, cfg)

    panel_moments = [
        inference_panel_moments(
            theta,
            solved,
            seed,
            cfg
        )
        for seed in seeds
    ]

    return np.mean(panel_moments, axis=0)

In [8]:
# Module 6: SMM Estimation and Weighting Matrix


def estimate_inference_stage(
    start,
    W,
    label,
    maxfev=MAX_FUNCTION_EVALUATIONS
):
    """
    Estimate the model parameters by minimizing the SMM objective.

    Compares the simulated moments to the target moments using weighting
    matrix W and searches for the parameter values that give the best fit.
    """
    W = np.asarray(W, dtype=float)

    # Check that weighting matrix has the correct dimensions
    if W.shape != (6, 6) or not np.isfinite(W).all():
        raise ValueError(
            "W must be a finite 6-by-6 matrix."
        )

    # Store objective values and previously evaluated parameter values
    history = []
    cache = {}

    # Keep track of the best parameter values found
    best = {
        "theta": np.array(start, dtype=float),
        "loss": np.inf
    }

    def raw_objective(theta):
        """
        Calculate the SMM loss for a given set of parameter values.
        """
        key = tuple(np.asarray(theta, dtype=float))

        # Return stored result if these parameters were already evaluated
        if key in cache:
            return cache[key]

        try:
            # Calculate simulated moments
            simulated_moments = inference_mean_moments(
                theta,
                ESTIMATION_SEEDS
            )

            # Difference between simulated and target moments
            moment_gap = (
                simulated_moments
                - np.asarray(target_moments)
            )

            # SMM objective function
            loss = float(
                moment_gap @ W @ moment_gap
            )

        except ValueError:
            # Give invalid parameter values a very large loss
            loss = 1e12

        cache[key] = loss

        # Update best parameter values
        if loss < best["loss"]:
            best.update(
                theta=np.array(theta).copy(),
                loss=loss
            )

        history.append(loss)

        # Print estimation progress
        if len(history) == 1 or len(history) % 25 == 0:
            print(
                f"{label}: evaluation {len(history)}, "
                f"best loss {best['loss']:.6f}",
                flush=True
            )

        return loss

    # Scale objective to improve numerical optimization
    scale = max(
        raw_objective(start),
        1e-8
    )

    def objective(theta):
        """
        Return the scaled SMM objective used by the optimizer.
        """
        return raw_objective(theta) / scale

    # First search for good parameter values using Powell
    exploration = minimize(
        objective,
        np.asarray(start, dtype=float),
        method="Powell",
        bounds=INFERENCE_BOUNDS,
        options=dict(
            maxfev=maxfev,
            xtol=1e-3,
            ftol=1e-3
        )
    )

    # Create starting simplex around the best parameter values found
    simplex = np.tile(
        best["theta"],
        (6, 1)
    )

    for j in range(5):
        step = FD_STEPS[j]

        if best["theta"][j] + step <= INFERENCE_BOUNDS[j, 1]:
            direction = 1
        else:
            direction = -1

        simplex[j + 1, j] += direction * step

    # Refine the best parameter values using Nelder-Mead
    fit = minimize(
        objective,
        best["theta"],
        method="Nelder-Mead",
        bounds=INFERENCE_BOUNDS,
        options=dict(
            maxfev=maxfev,
            initial_simplex=simplex,
            xatol=1e-4,
            fatol=1e-6
        )
    )

    # Check whether the optimizer finished at the best solution found
    terminal_loss = raw_objective(fit.x)

    terminal_is_best = (
        abs(terminal_loss - best["loss"])
        <= 1e-6 * max(1.0, best["loss"])
    )

    converged = bool(
        fit.success and terminal_is_best
    )

    print(
        f"{label}: converged={converged}; {fit.message}",
        flush=True
    )

    if best["loss"] >= 1e12:
        raise ValueError(
            "No valid parameter trial found."
        )

    return {
        "theta": best["theta"],
        "loss": best["loss"],
        "optimizer": fit,
        "exploration": exploration,
        "converged": converged,
        "history": history
    }


def estimate_moment_covariance(
    theta,
    seeds=COVARIANCE_SEEDS,
    cfg=INFERENCE_CONFIG
):
    """
    Estimate the covariance matrix of the six simulated moments.

    Simulates independent panels using different seeds and calculates
    the covariance of their moments and the SMM weighting matrix.
    """
    # Make sure there are enough independent simulation draws
    if len(seeds) <= 6 or len(set(seeds)) != len(seeds):
        raise ValueError(
            "Use more than six distinct independent panel seeds."
        )

    # Solve the model once for the given parameter values
    solved = solve_inference_dp(
        theta,
        cfg
    )

    panel_moments = []

    # Simulate independent panels and calculate moments
    for i, seed in enumerate(seeds, start=1):

        moments = inference_panel_moments(
            theta,
            solved,
            seed,
            cfg
        )

        panel_moments.append(moments)

        if i % 25 == 0:
            print(
                f"Covariance panels: {i}/{len(seeds)}",
                flush=True
            )

    panel_moments = np.asarray(panel_moments)

    # Calculate covariance matrix of simulated moments
    Omega = np.cov(
        panel_moments,
        rowvar=False,
        ddof=1
    )

    # Check that covariance matrix can be inverted
    eigenvalues = np.linalg.eigvalsh(Omega)

    if (
        eigenvalues[0] <= 0
        or np.linalg.cond(Omega) > 1e12
    ):
        raise ValueError(
            "Moment covariance is singular or ill conditioned; "
            "increase panels or diagnose redundant moments."
        )

    # Optimal SMM weighting matrix is the inverse of moment covariance
    W_star = np.linalg.solve(
        Omega,
        np.eye(6)
    )

    # Make weighting matrix exactly symmetric
    W_star = (
        W_star + W_star.T
    ) / 2

    return panel_moments, Omega, W_star

In [9]:
# Module 7: Numerical Derivatives and Standard Errors

def numerical_moment_jacobian(
    theta,
    steps=FD_STEPS,
    seeds=DERIVATIVE_SEEDS
):
    """
    Calculate the Jacobian of the six moments with respect to the five parameters.

    Uses finite differences to measure how each simulated moment changes
    when each parameter is changed by a small amount.
    """
    theta = np.asarray(theta, dtype=float)

    # Calculate moments at estimated parameter values
    base_moments = inference_mean_moments(
        theta,
        seeds
    )

    # Six moments and five parameters
    D = np.empty((6, 5))

    # Keep track of finite difference method used for each parameter
    methods = []

    # Change one parameter at a time
    for j, step in enumerate(steps):

        lower_bound, upper_bound = INFERENCE_BOUNDS[j]

        theta_plus = theta.copy()
        theta_minus = theta.copy()

        theta_plus[j] += step
        theta_minus[j] -= step

        # Use central difference when both directions are within bounds
        if (
            theta_minus[j] >= lower_bound
            and theta_plus[j] <= upper_bound
        ):
            moments_plus = inference_mean_moments(
                theta_plus,
                seeds
            )

            moments_minus = inference_mean_moments(
                theta_minus,
                seeds
            )

            D[:, j] = (
                moments_plus - moments_minus
            ) / (2 * step)

            methods.append("central")

        # Use forward difference near the lower bound
        elif theta_plus[j] <= upper_bound:

            moments_plus = inference_mean_moments(
                theta_plus,
                seeds
            )

            D[:, j] = (
                moments_plus - base_moments
            ) / step

            methods.append("forward")

        # Use backward difference near the upper bound
        elif theta_minus[j] >= lower_bound:

            moments_minus = inference_mean_moments(
                theta_minus,
                seeds
            )

            D[:, j] = (
                base_moments - moments_minus
            ) / step

            methods.append("backward")

        else:
            raise ValueError(
                "Finite-difference step is larger than "
                "the allowed parameter interval."
            )

    return D, methods


def standard_errors_from_jacobian(
    D,
    W_star,
    S=S_ESTIMATION
):
    """
    Calculate parameter standard errors using the moment Jacobian.

    Uses the Jacobian and optimal SMM weighting matrix to calculate
    the covariance matrix and standard errors of the estimated parameters.
    """
    # Check dimensions
    if D.shape != (6, 5) or S < 1:
        raise ValueError(
            "Need a 6-by-5 Jacobian and S >= 1."
        )

    # Check that the five parameters are identified by the moments
    whitened = (
        np.linalg.cholesky(W_star).T @ D
    )

    if np.linalg.matrix_rank(whitened) < 5:
        raise ValueError(
            "Moment Jacobian has rank below 5: "
            "ordinary standard errors are not identified on this grid."
        )

    # Calculate information matrix
    information = (
        D.T @ W_star @ D
    )

    # Make matrix exactly symmetric
    information = (
        information + information.T
    ) / 2

    # Check that information matrix can be reliably inverted
    if np.linalg.cond(information) > 1e12:
        raise ValueError(
            "D' W D is too ill conditioned "
            "for reliable standard errors."
        )

    # Calculate covariance matrix of estimated parameters
    covariance_theta = (
        (1 + 1 / S)
        * np.linalg.solve(
            information,
            np.eye(5)
        )
    )

    # Make covariance matrix exactly symmetric
    covariance_theta = (
        covariance_theta + covariance_theta.T
    ) / 2

    # Check that parameter variances are positive
    if np.any(np.diag(covariance_theta) <= 0):
        raise ValueError(
            "Parameter covariance has nonpositive diagonal entries."
        )

    # Standard errors are square roots of parameter variances
    standard_errors = np.sqrt(
        np.diag(covariance_theta)
    )

    return covariance_theta, standard_errors

panel_moments, Omega, W_star = estimate_moment_covariance(
    theta_stage1
)


NameError: name 'theta_stage1' is not defined

In [ ]:
# Module 8: First-Stage SMM Estimation

# Starting parameter values for SMM estimation
theta_start = np.array(
    [alpha, gamma, rho, sigma, phi_tilde0],
    dtype=float
)

# First-stage estimation uses the identity weighting matrix
W_identity = np.eye(6)

# Estimate parameters
stage1 = estimate_inference_stage(
    theta_start,
    W_identity,
    "Identity stage"
)

# Store first-stage parameter estimates
theta_stage1 = stage1["theta"]

# Display results
print(
    "Stage-1 estimates:",
    dict(zip(PARAMETER_NAMES, theta_stage1))
)

print(
    "Stage-1 loss:",
    stage1["loss"]
)

print(
    "Stage-1 converged:",
    stage1["converged"]
)

Identity stage: evaluation 1, best loss 0.331009
Identity stage: evaluation 25, best loss 0.276967
Identity stage: evaluation 50, best loss 0.188358
Identity stage: evaluation 75, best loss 0.185588
Identity stage: evaluation 100, best loss 0.123693
Identity stage: evaluation 125, best loss 0.112608
Identity stage: evaluation 150, best loss 0.092775
Identity stage: evaluation 175, best loss 0.068014
Identity stage: evaluation 200, best loss 0.067496
Identity stage: evaluation 225, best loss 0.067496
Identity stage: evaluation 250, best loss 0.067496
Identity stage: evaluation 275, best loss 0.067496
Identity stage: evaluation 300, best loss 0.067448
Identity stage: evaluation 325, best loss 0.067001
Identity stage: evaluation 350, best loss 0.066952
Identity stage: evaluation 375, best loss 0.066901
Identity stage: evaluation 400, best loss 0.066843
Identity stage: evaluation 425, best loss 0.066782
Identity stage: evaluation 450, best loss 0.066779
Identity stage: evaluation 475, best

In [ ]:
# Module 9: Second-Stage SMM Estimation

# Estimate parameters using the optimal weighting matrix
stage2 = estimate_inference_stage(
    theta_stage1,
    W_star,
    "Efficient stage"
)

# Store second-stage parameter estimates
theta_stage2 = stage2["theta"]

# Calculate simulated moments at second-stage estimates
moments_stage2 = inference_mean_moments(
    theta_stage2,
    ESTIMATION_SEEDS
)

# Display parameter estimates
print(
    "Stage-2 estimates:",
    dict(zip(PARAMETER_NAMES, theta_stage2))
)

print(
    "Stage-2 converged:",
    stage2["converged"]
)

print(
    "Stage-2 loss:",
    stage2["loss"]
)

# Compare target moments with second-stage simulated moments
moment_comparison = pd.DataFrame(
    {
        "Target": target_moments,
        "Stage 2": moments_stage2
    },
    index=MOMENT_NAMES
)

print(
    moment_comparison.to_string(
        float_format=lambda x: f"{x:.4f}"
    )
)

Efficient stage: evaluation 1, best loss 17906.567856
Efficient stage: evaluation 25, best loss 17906.567856
Efficient stage: evaluation 50, best loss 17906.567856
Efficient stage: evaluation 75, best loss 17906.567856
Efficient stage: evaluation 100, best loss 17906.567856
Efficient stage: evaluation 125, best loss 17906.567856
Efficient stage: evaluation 150, best loss 17823.406508
Efficient stage: evaluation 175, best loss 17809.724939
Efficient stage: evaluation 200, best loss 17796.490036
Efficient stage: evaluation 225, best loss 17790.774050
Efficient stage: evaluation 250, best loss 17789.912755
Efficient stage: evaluation 275, best loss 17788.750237
Efficient stage: evaluation 300, best loss 17785.647592
Efficient stage: evaluation 325, best loss 17762.578691
Efficient stage: evaluation 350, best loss 17761.405699
Efficient stage: evaluation 375, best loss 17761.350177
Efficient stage: evaluation 400, best loss 17761.264089
Efficient stage: evaluation 425, best loss 17761.2268

In [ ]:
# Module 10: Standard Errors and Inference Diagnostics

# Calculate Jacobian using the original finite-difference steps
jacobian, derivative_methods = numerical_moment_jacobian(
    theta_stage2,
    FD_STEPS
)

# Calculate Jacobian again using smaller steps
jacobian_half, derivative_methods_half = numerical_moment_jacobian(
    theta_stage2,
    FD_STEPS / 2
)

# Calculate parameter covariance matrix and standard errors
parameter_covariance, standard_errors = standard_errors_from_jacobian(
    jacobian,
    W_star,
    S_ESTIMATION
)

parameter_covariance_half, standard_errors_half = standard_errors_from_jacobian(
    jacobian_half,
    W_star,
    S_ESTIMATION
)

# Compare derivatives using the two step sizes
relative_derivative_change = (
    np.linalg.norm(
        jacobian - jacobian_half,
        axis=0
    )
    / np.maximum(
        np.linalg.norm(jacobian_half, axis=0),
        1e-12
    )
)

# Check whether any estimated parameters are close to their bounds
near_bound = (
    np.minimum(
        theta_stage2 - INFERENCE_BOUNDS[:, 0],
        INFERENCE_BOUNDS[:, 1] - theta_stage2
    )
    < 1e-4
)

# Check whether derivatives are stable across step sizes
stable_derivatives = bool(
    np.all(relative_derivative_change < 0.25)
)

# Check whether numerical inference diagnostics pass
inference_checks_pass = bool(
    stage1["converged"]
    and stage2["converged"]
    and stable_derivatives
    and not np.any(near_bound)
)

# Create table of parameter estimates and standard errors
standard_errors_table = pd.DataFrame(
    {
        "Identity estimate": theta_stage1,
        "Efficient estimate": theta_stage2,
        "SE (h)": standard_errors,
        "SE (h/2)": standard_errors_half,
        "Derivative change": relative_derivative_change,
        "Difference method": derivative_methods,
        "At bound": near_bound
    },
    index=PARAMETER_NAMES
)

# Display parameter estimates and inference results
print(
    standard_errors_table.to_string(
        float_format=lambda x: f"{x:.4f}"
    )
)

# Display numerical diagnostic results
if inference_checks_pass:
    print("Status: Numerical checks passed")
else:
    print(
        "Status: PROVISIONAL: review convergence, "
        "derivative sensitivity and boundary diagnostics"
    )

print(
    f"S={S_ESTIMATION}; "
    f"R={R_COVARIANCE}; "
    f"simulation adjustment={1 + 1 / S_ESTIMATION:.4f}"
)

print(
    "Jacobian shape:",
    jacobian.shape
)

print(
    "Jacobian rank:",
    np.linalg.matrix_rank(jacobian)
)

print(
    "Information matrix condition number:",
    np.linalg.cond(
        jacobian.T @ W_star @ jacobian
    )
)

            Identity estimate  Efficient estimate  SE (h)  SE (h/2)  Derivative change Difference method  At bound
alpha                  0.7021              0.7020  0.0032    0.0003             0.9140           central     False
gamma                  0.9690              0.9691  0.0042    0.0009             0.5104           central     False
rho                    0.4065              0.4069  0.0088    0.0065             0.5468           central     False
sigma                  0.4468              0.4476  0.0021    0.0016             0.6644           central     False
phi_tilde0             1.2352              1.2321  0.0133    0.0121             0.4970           central     False
Status: PROVISIONAL: review convergence, derivative sensitivity and boundary diagnostics
S=1; R=100; simulation adjustment=2.0000
Jacobian shape: (6, 5)
Jacobian rank: 5
Information matrix condition number: 876.9173994042036
